# Visualize the agent's memory graph

An interactive, Neo4j-Browser-like view of everything the agent remembers, rendered
inline with [`yfiles-jupyter-graphs-for-neo4j`](https://github.com/yWorks/yfiles-jupyter-graphs-for-neo4j).
Pan, zoom, click nodes to inspect properties, and drag the layout around.

Node labels in this graph (see the schema sketch in the README):
`Message`, `GameSnapshot`, reasoning `Trace` / `Step` / `ToolCall`, and the long-term
`Entity` / `Preference` semantic model.

Prereqs:
- Neo4j is up (`bash scripts/vast_neo4j_launch.sh`, or `docker compose up -d neo4j`).
- `pip install -r requirements.txt` (installs `yfiles-jupyter-graphs-for-neo4j`).

In [ ]:
import os
import sys

# Run from the repo root so `agent` imports and `.env` resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from neo4j import GraphDatabase
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget

from agent.config import CONFIG

driver = GraphDatabase.driver(
    CONFIG.neo4j_uri,
    auth=(CONFIG.neo4j_username, CONFIG.neo4j_password),
    database=CONFIG.neo4j_database,
)

g = Neo4jGraphWidget(driver)
# Auto-draw relationships between the nodes a query returns, even if the query
# itself only returned the nodes.
g.set_autocomplete_relationships(True)

## Per-label captions & colors

Give each node a readable caption and a distinct color so the graph is legible.
The `text` binding receives the element dict; node properties live under
`element["properties"]`.

In [ ]:
def _prop(element, key, default=""):
    return element.get("properties", {}).get(key, default)


g.add_node_configuration(
    "Message",
    color="#4C78A8",
    text=lambda n: f"{_prop(n, 'role', '?')}: {str(_prop(n, 'content'))[:40]}",
)
g.add_node_configuration(
    "GameSnapshot",
    color="#F58518",
    text=lambda n: f"snapshot [{_prop(n, 'label', 'step')}]",
)
g.add_node_configuration(
    "Trace", color="#54A24B", text=lambda n: f"Trace: {str(_prop(n, 'task'))[:40]}"
)
g.add_node_configuration(
    "Step", color="#88D27A", text=lambda n: f"Step: {str(_prop(n, 'thought'))[:40]}"
)
g.add_node_configuration(
    "ToolCall", color="#B79A20", text=lambda n: f"tool: {_prop(n, 'tool')}"
)
g.add_node_configuration(
    "Entity", color="#E45756", text=lambda n: f"{_prop(n, 'name')} ({_prop(n, 'type')})"
)
g.add_node_configuration(
    "Preference",
    color="#72B7B2",
    text=lambda n: f"pref: {_prop(n, 'category')}",
)

## The whole memory graph

Everything currently stored. If the graph is large, bump/lower the `LIMIT`.

In [ ]:
g.show_cypher("MATCH (n)-[r]-(m) RETURN n, r, m LIMIT 500", layout="organic")

## One conversation / session

Scroll through a single session's experiences: its messages, the game snapshots
they captured, and the reasoning traces behind each move. Set `SESSION_ID` to the
`session_id` printed by the play notebook or `agent.runner`.

In [ ]:
SESSION_ID = ""  # e.g. the session_id printed by notebooks/play.ipynb

if SESSION_ID:
    g.show_cypher(
        "MATCH (n {session_id: $sid}) "
        "OPTIONAL MATCH (n)-[r]-(m) "
        "RETURN n, r, m",
        sid=SESSION_ID,
        layout="hierarchic",
    )
else:
    print("Set SESSION_ID above to scope the view to one conversation.")

When you're done:

In [ ]:
driver.close()